In [ ]:
from utils import prepare_dataframe, merge_csv_files_in_directory

merge_csv_files_in_directory()
df = prepare_dataframe("merged.csv")
resulting_label_count = df['labels'].nunique()
df


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "microsoft/graphcodebert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("cuda:0")

In [ ]:
import os

embedding_directory = 'embeddings/GCB_LHS_embeddings_patch_before_largewithcodedata_15300limit3label'

os.makedirs(embedding_directory, exist_ok=True)

In [ ]:
from utils import extract_code_from_patch

# Tokenizing using subtracting full code before and after embeddings

# def generate_and_save_embeddings_batch(contents_before, contents_after, indices):
def generate_and_save_embedding(patch,label, index):

    patch_before, patch_after = extract_code_from_patch(patch)
    
    
    # Tokenize the before and after code patch snippet
    tokens_before = tokenizer(patch_before, truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Tokenize the before and after code snippets
    # tokens_before = tokenizer(list(contents_before), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")
    # tokens_after = tokenizer(list(contents_after), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Get the embeddings for both tokenized sequences
    with torch.no_grad():

        embedding_before = model(**tokens_before).last_hidden_state
        # embeddings_after = model(**tokens_after).pooler_output
    
    # Subtract the embeddings
    # subtracted_embeddings = embeddings_after - embeddings_before
    # print(subtracted_embeddings.shape)
    
    # Average the subtracted embeddings across the token dimension
    # subtracted_embeddings = subtracted_embeddings.mean(dim=1)
    
    save_path = os.path.join(embedding_directory, f"embedding_{index}.pt")
    torch.save(embedding_before, save_path)
    

In [ ]:
import tqdm

for index,row in tqdm.notebook.tqdm(df.iterrows()):
    generate_and_save_embedding(row['patch'],row['labels'],row['index'])